In [1]:
import os
os.environ["LANGCHAIN_TRACING_V2"] = "false"

In [2]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "BAAI/bge-m3"
persist_dir = "../../data/vector_db"
collection_name = "test"   


embedding = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device": "cuda"} 
)


vector_db = Chroma(
    persist_directory=persist_dir,
    embedding_function=embedding,
    collection_name=collection_name
)
retriever = vector_db.as_retriever(search_kwargs={"k": 5})

query = "Công thức hàm đối ngẫu Lagrange là gì"

docs = retriever.invoke(query)

d:\VSCode\AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from transformers import AutoModel

model = AutoModel.from_pretrained(
    'jinaai/jina-reranker-v3',
    dtype="auto",
    trust_remote_code=True,
)
model.to('cuda') # or 'cpu' if no GPU is available
model.eval()


# Rerank documents
rerank_docs = model.rerank(
    query,
    [doc.page_content for doc in docs])

In [4]:
from dotenv import load_dotenv

load_dotenv()

print(f"API_KEY: {os.getenv("DEEPSEEK_API_KEY")[:10]}")

API_KEY: sk-ac963a3


In [5]:
# prompt
from langchain_core.prompts import ChatPromptTemplate
template_ = """
You are an AI assistant that answers questions based ONLY on the provided context.

GENERAL RULES:
- Use only the information from the CONTEXT.
- If the answer is not in the context, say: "I don't have enough information."
- Be clear, accurate, and well-structured.
- Avoid unnecessary repetition or verbosity.

ADAPTIVE RESPONSE STYLE:
- If the question asks for a definition → give a concise definition first, then explain.
- If the question involves formulas or mathematics → include properly formatted LaTeX using $$ $$.
- If the question is conceptual → provide a clear explanation with logical structure.
- If the question is procedural → answer step-by-step.
- If multiple points are needed → use bullet points.

FORMATTING:
- Use paragraphs for explanations.
- Use bullet points when listing ideas.
- Use LaTeX only when necessary (not for every answer).

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

prompt_ = ChatPromptTemplate.from_template(template_, verbose=True)

In [6]:
# chat model
from langchain_deepseek import ChatDeepSeek

llm = ChatDeepSeek(model='deepseek-chat', temperature=0.0)

# chain
chain = prompt_ | llm

# invoke Langchain's chain
results = chain.invoke({"context": rerank_docs, "question": query})
print(results.content)

Công thức hàm đối ngẫu Lagrange (Lagrange dual function) được định nghĩa là:

$$g(\boldsymbol{\lambda}, \boldsymbol{\nu}) = \inf_{\mathbf{x} \in \mathcal{D}} \mathcal{L}(\mathbf{x}, \boldsymbol{\lambda}, \boldsymbol{\nu}) = \inf_{\mathbf{x} \in \mathcal{D}} \left( f_0(\mathbf{x}) + \sum_{i=1}^m \lambda_i f_i(\mathbf{x}) + \sum_{j=1}^p \nu_j h_j(\mathbf{x}) \right)$$

Trong đó:
- $\mathcal{D}$ là tập xác định của bài toán tối ưu chính.
- $\mathcal{L}(\mathbf{x}, \boldsymbol{\lambda}, \boldsymbol{\nu})$ là hàm Lagrange.
- $\lambda_i \ge 0$ (với $i=1,\dots,m$) và $\nu_j$ (với $j=1,\dots,p$) là các biến đối ngẫu (nhân tử Lagrange).
- Nếu hàm Lagrange không bị chặn dưới, hàm đối ngẫu nhận giá trị $-\infty$.


In [8]:
from IPython.display import display, Markdown

display(Markdown(results.content))

Công thức hàm đối ngẫu Lagrange (Lagrange dual function) được định nghĩa là:

$$g(\boldsymbol{\lambda}, \boldsymbol{\nu}) = \inf_{\mathbf{x} \in \mathcal{D}} \mathcal{L}(\mathbf{x}, \boldsymbol{\lambda}, \boldsymbol{\nu}) = \inf_{\mathbf{x} \in \mathcal{D}} \left( f_0(\mathbf{x}) + \sum_{i=1}^m \lambda_i f_i(\mathbf{x}) + \sum_{j=1}^p \nu_j h_j(\mathbf{x}) \right)$$

Trong đó:
- $\mathcal{D}$ là tập xác định của bài toán tối ưu chính.
- $\mathcal{L}(\mathbf{x}, \boldsymbol{\lambda}, \boldsymbol{\nu})$ là hàm Lagrange.
- $\lambda_i \ge 0$ (với $i=1,\dots,m$) và $\nu_j$ (với $j=1,\dots,p$) là các biến đối ngẫu (nhân tử Lagrange).
- Nếu hàm Lagrange không bị chặn dưới, hàm đối ngẫu nhận giá trị $-\infty$.